---
title: "Poza średnią"
---

Średnia jest wygodna, ale potrafi być zdradliwa. Jeżeli pokażemy tylko jeden
słupek z przeciętną wartością, ukryjemy rozrzut, asymetrię i obserwacje odstające.
Dlatego przed interpretacją warto zobaczyć cały rozkład.

In [ ]:
#| label: setup-19
library(tidyverse)
library(here)
library(janitor)

source(here("R", "theme_course.R"))
theme_set(theme_course())

## Przygotowanie danych

W pliku `student-alcohol-consumption.csv` mamy m.in. ocenę końcową `G3` oraz
poziom weekendowego spożycia alkoholu `Walc` w skali od 1 do 5. To dobry materiał
do pokazania, że podobne średnie nie muszą oznaczać podobnych rozkładów.

In [ ]:
#| label: data-prep-19
students <- readr::read_csv(
  here("datasets", "student-alcohol-consumption.csv"),
  show_col_types = FALSE
) |>
  janitor::clean_names() |>
  transmute(
    walc = factor(
      walc,
      levels = 1:5,
      labels = c("1 niskie", "2", "3", "4", "5 wysokie")
    ),
    g3
  ) |>
  drop_na(walc, g3)

mean_scores <- students |>
  group_by(walc) |>
  summarise(mean_g3 = mean(g3), .groups = "drop")

## Wersja uproszczona: same średnie

Średnie są tutaj do siebie zaskakująco podobne. Problem w tym, że taki wykres
nie mówi nic o tym, czy grupy są stabilne, szerokie, wielomodalne albo pełne
wartości odstających.

In [ ]:
#| label: fig-mean-bars-students
#| fig-cap: "Wykres średnich ukrywa kształt rozkładu ocen końcowych."
#| fig-alt: "Wykres słupkowy pokazuje średnią ocenę końcową dla pięciu poziomów weekendowego spożycia alkoholu. Średnie są do siebie zbliżone, więc sam wykres sugeruje niewielkie różnice."
ggplot(mean_scores, aes(x = walc, y = mean_g3)) +
  geom_col(fill = "#6C9A8B", width = 0.72) +
  geom_text(aes(label = round(mean_g3, 1)), vjust = -0.5, size = 3.5) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.08))) +
  labs(
    title = "Średnia wygląda bezpiecznie, ale mówi bardzo mało",
    subtitle = "Ocena końcowa G3 według poziomu weekendowego spożycia alkoholu",
    x = "Weekendowe spożycie alkoholu",
    y = "Średnia ocena końcowa",
    caption = "Źródło: datasets/student-alcohol-consumption.csv"
  )

## Naprawa: pokaż cały rozkład

Po dodaniu wykresu skrzypcowego, pudełkowego i surowych punktów od razu widać,
że podobne średnie mogą maskować zupełnie inną zmienność i gęstość obserwacji.

In [ ]:
#| label: fig-violin-box-jitter-students
#| fig-cap: "Połączenie violin, boxplot i jitter odsłania rozkład, medianę i zagęszczenie punktów."
#| fig-alt: "Dla każdej kategorii weekendowego spożycia alkoholu pokazano skrzypce, pudełko i pojedyncze punkty ocen końcowych. Widać pełny rozkład, rozrzut i miejsca koncentracji danych."
ggplot(students, aes(x = walc, y = g3, fill = walc)) +
  geom_violin(trim = FALSE, alpha = 0.78, color = NA) +
  geom_boxplot(
    width = 0.14,
    fill = "white",
    color = "#1f2933",
    outlier.shape = NA,
    linewidth = 0.45
  ) +
  geom_jitter(
    width = 0.12,
    alpha = 0.18,
    size = 1.2,
    color = "#1f2933"
  ) +
  scale_fill_viridis_d(option = "C", end = 0.9, guide = "none") +
  scale_y_continuous(breaks = seq(0, 20, 2)) +
  labs(
    title = "Rozkład pokazuje ryzyko, którego średnia nie ujawnia",
    subtitle = "Te grupy mają podobne średnie, ale nie mają identycznej zmienności",
    x = "Weekendowe spożycie alkoholu",
    y = "Ocena końcowa G3",
    caption = "Źródło: datasets/student-alcohol-consumption.csv"
  )

## Wnioski i interpretacja

Kiedy pokazujesz zmienność, pokazujesz też uczciwość analityczną. Odbiorca widzi
nie tylko jedną liczbę, ale cały zakres niepewności i strukturę danych, na których
opiera się wniosek.

## Zadanie

Powtórz tę analizę dla innej zmiennej kategorycznej z tego zbioru, na przykład
`study_time`, `internet` albo `school`, i sprawdź, czy sam boxplot wystarczy,
czy nadal warto dodać skrzypce i punkty.